In [127]:
import polars as pl
from scipy.stats import fisher_exact
from pathlib import Path
import os
from tqdm.auto import tqdm

os.chdir('/home/vladimirnoz/Projects/Codebook_Perspectives/AS_CHS_GHTS')

TABLES_PATH = Path('joint_tables/')
SIGNIF_THR = 0.01

In [128]:
tables_list = []

for path in tqdm(sorted(TABLES_PATH.glob('ASB_Phenotypes/*.tsv'))):
    local_df = pl.read_csv(
        path, 
        separator='\t',
        schema_overrides = {
            'bads': pl.String,
            'ref_counts': pl.String,
            'alt_counts': pl.String,
            'ref_es': pl.String,
            'alt_es': pl.String,
            'ref_pval': pl.String,
            'alt_pval': pl.String,
        }
    )
    tf = path.stem
    exp = path.parent.parent.stem
    local_df = local_df.with_columns(
        tf = pl.lit(tf), 
        exp = pl.lit(exp),
    )
    tables_list.append(local_df)
df = pl.concat(tables_list, how="vertical_relaxed")
df

  0%|          | 0/1 [00:00<?, ?it/s]

#chr,start,end,mean_bad,id,max_cover,ref,alt,n_reps,ref_comb_es,alt_comb_es,ref_comb_pval,alt_comb_pval,ref_fdr_comb_pval,alt_fdr_comb_pval,pref_allele,comb_es,comb_pval,fdr_comb_pval,ebi,eQTL_cis,eQTL_cis_gene,ADASTRA_TF,ADASTRA_CL,tf,exp
str,i64,i64,f64,str,i64,str,str,i64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,str,str,str,str,str,str,str
"""chr1""",17275695,17275696,2.0,"""rs3003434""",25,"""A""","""G""",2,1.035011,-1.4577,0.021948,0.82837,0.164123,0.98799,"""ref""",1.035011,0.021948,0.164123,"""-""","""Lung""","""ENSG00000159339.13""","""ZN350_HUMAN;BRD4_HUMAN;PBX4_HU…","""primary_human_epidermal_kerati…","""selex+chipseq""","""joint_tables"""
"""chr1""",17275729,17275730,2.0,"""rs12135460""",23,"""C""","""T""",2,0.790578,-1.126043,0.126463,0.789824,0.427968,0.967883,"""ref""",0.790578,0.126463,0.427968,"""-""","""-""","""-""","""ZN350_HUMAN;ZN776_HUMAN""","""HEK293__embryonic_kidney_""","""selex+chipseq""","""joint_tables"""
"""chr1""",44731028,44731029,2.0,"""rs58532407""",41,"""T""","""C""",16,-0.611462,0.482598,0.72177,0.023907,0.939801,0.171455,"""alt""",0.482598,0.023907,0.171455,"""-""","""Adipose_Subcutaneous;Adipose_V…","""ENSG00000117410.13;ENSG0000014…","""MIER1_HUMAN;TF65_HUMAN;ZN610_H…","""thyroid_gland;LoVo__colorectal…","""selex+chipseq""","""joint_tables"""
"""chr1""",44731045,44731046,2.0,"""rs7524446""",47,"""C""","""A""",17,-0.556663,0.398918,0.598453,0.038888,0.862237,0.227995,"""alt""",0.398918,0.038888,0.227995,"""-""","""Adipose_Subcutaneous;Artery_Ti…","""ENSG00000070785.16;ENSG0000012…","""FOXK2_HUMAN;BRD9_HUMAN;SMAD2_H…","""HAEC__human_aortic_endothelial…","""selex+chipseq""","""joint_tables"""
"""chr1""",55335837,55335838,2.0,"""rs74979685""",35,"""G""","""A""",2,0.559125,-0.38518,0.239005,0.412974,0.580708,0.72932,"""ref""",0.559125,0.239005,0.580708,"""-""","""-""","""-""","""-""","""-""","""selex+chipseq""","""joint_tables"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr2""",234060729,234060730,3.0,"""rs10195232""",39,"""T""","""C""",1,-2.2704,1.354413,0.899231,0.036279,1.0,0.220942,"""alt""",1.354413,0.036279,0.220942,"""-""","""-""","""-""","""ZNF24_HUMAN;SSRP1_HUMAN;ZN207_…","""HepG2__hepatoblastoma_;HT1080_…","""selex+chipseq""","""joint_tables"""
"""chr4""",49315259,49315260,3.0,"""rs9799445""",30,"""A""","""C""",2,-2.000153,0.8801,0.887183,0.063231,1.0,0.296764,"""alt""",0.8801,0.063231,0.296764,"""-""","""-""","""-""","""-""","""-""","""selex+chipseq""","""joint_tables"""
"""chr6""",170163845,170163846,2.0,"""rs566969713""",27,"""G""","""C""",1,-0.484538,-0.087276,0.463999,0.316769,0.761747,0.652908,"""alt""",-0.087276,0.316769,0.652908,"""-""","""-""","""-""","""-""","""-""","""selex+chipseq""","""joint_tables"""


In [129]:
print(len(df))
print(len(df.filter(pl.col('fdr_comb_pval') < 0.05)))

143013
22064


In [134]:
calc = df.lazy().with_columns(
    signif = pl.col('fdr_comb_pval') < SIGNIF_THR
)

phenotypes = [
     'ebi',
     'eQTL_cis',
     'ADASTRA_TF',
]

calc = calc.with_columns(
    [(pl.col(x) != '-').alias(x) for x in phenotypes]
    
).group_by(
    [
        '#chr', 
        'start', 
        'end', 
        'id', 
        'ref', 
        'alt',
    ]
).agg(
   [pl.col(x).max().alias(x) for x in phenotypes + ['signif']]
)



In [135]:
aggs = [
    pl.col("signif").sum().alias("tot_sig"),
    (~pl.col("signif")).sum().alias("tot_not_sig")
]

for p in phenotypes:
    aggs.append((pl.col(p) & pl.col("signif")).sum().alias(f"{p}_n1"))
    aggs.append((pl.col(p) & ~pl.col("signif")).sum().alias(f"{p}_n2"))

counts = calc.select(aggs).collect()
print(counts)
total_sig = counts["tot_sig"][0]
total_not_sig = counts["tot_not_sig"][0]

results = []

for p in phenotypes:
    n1 = counts[f"{p}_n1"][0]
    n2 = counts[f"{p}_n2"][0]
    n3 = total_sig - n1
    n4 = total_not_sig - n2
    
    table = [[n1, n2], [n3, n4]]
    odds, pval = fisher_exact(table)
    
    results.append({
        "phenotype": p,
        "N1 (P+ S+)": n1,
        "N2 (P+ S-)": n2,
        "N3 (P- S+)": n3,
        "N4 (P- S-)": n4,
        "odds_ratio": odds,
        "p_value": pval,
        "signif_%": f'{n1/(n1+n3):.0%}'
    })

final_stats = pl.DataFrame(results)
final_stats

shape: (1, 8)
┌─────────┬─────────────┬────────┬────────┬─────────────┬─────────────┬──────────────┬─────────────┐
│ tot_sig ┆ tot_not_sig ┆ ebi_n1 ┆ ebi_n2 ┆ eQTL_cis_n1 ┆ eQTL_cis_n2 ┆ ADASTRA_TF_n ┆ ADASTRA_TF_ │
│ ---     ┆ ---         ┆ ---    ┆ ---    ┆ ---         ┆ ---         ┆ 1            ┆ n2          │
│ u32     ┆ u32         ┆ u32    ┆ u32    ┆ u32         ┆ u32         ┆ ---          ┆ ---         │
│         ┆             ┆        ┆        ┆             ┆             ┆ u32          ┆ u32         │
╞═════════╪═════════════╪════════╪════════╪═════════════╪═════════════╪══════════════╪═════════════╡
│ 11142   ┆ 111222      ┆ 1144   ┆ 8716   ┆ 7973        ┆ 72608       ┆ 10272        ┆ 92358       │
└─────────┴─────────────┴────────┴────────┴─────────────┴─────────────┴──────────────┴─────────────┘


phenotype,N1 (P+ S+),N2 (P+ S-),N3 (P- S+),N4 (P- S-),odds_ratio,p_value,signif_%
str,i64,i64,i64,i64,f64,f64,str
"""ebi""",1144,8716,9998,102506,1.34569,3.7921e-18,"""10%"""
"""eQTL_cis""",7973,72608,3169,38614,1.338011,1.7686e-41,"""72%"""
"""ADASTRA_TF""",10272,92358,870,18864,2.411543,3.8992e-162,"""92%"""
